# Hệ thống Gợi ý Việc làm & Ứng viên (AI Server Logic) - OPTIMIZED v2

### Kiến trúc hệ thống
Notebook này chứa logic cốt lõi cho **Server AI**, một microservice riêng biệt trong toàn bộ hệ thống.
- **Server Spring Boot (Base Server)**: Xử lý logic nghiệp vụ, quản lý DB chính (SQL).
- **Server AI (Python - Logic trong Notebook này)**: Xử lý các tác vụ AI, giao tiếp với Vector DB.
- **ChromaDB**: Vector Database, phục vụ tìm kiếm ngữ nghĩa tốc độ cao.

### Vai trò của Notebook
Notebook này là môi trường để phát triển, thử nghiệm và tinh chỉnh các thuật toán AI trước khi triển khai thành các API thực tế (sử dụng Flask/FastAPI).
Phiên bản này đã được tối ưu hóa về thuật toán tính điểm và độ ổn định.

## 1. Setup & Import Libraries

In [58]:
import pandas as pd
import numpy as np
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize
import re
import torch
import os
import shutil
import warnings
from typing import Dict, List, Any

warnings.filterwarnings('ignore')

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 2. Load Data & Preprocessing

**Ghi chú trong kiến trúc thực tế:**
- Bước này mô phỏng việc **Server AI nhận dữ liệu từ Server Spring Boot** qua một API request.
- Dữ liệu gốc (từ file CSV) thực chất được lưu trong Database SQL (PostgreSQL/MySQL) và do Spring Boot quản lý.

In [59]:
# 2.1 Load Data
candidates = pd.read_csv("../data/csv_data/candidates.csv")
users = pd.read_csv("../data/csv_data/users.csv")
candidates_skills = pd.read_csv("../data/csv_data/candidates_skills.csv")
skills = pd.read_csv("../data/csv_data/skills.csv")

job_posts = pd.read_csv("../data/csv_data/job_posts.csv")
job_posts_skills = pd.read_csv("../data/csv_data/job_posts_skills.csv")
industries = pd.read_csv("../data/csv_data/industries.csv")
levels = pd.read_csv("../data/csv_data/levels.csv")
job_types = pd.read_csv("../data/csv_data/job_types.csv")

# --- MERGE DATA FOR CANDIDATES ---
candidates_full = candidates.merge(users[['id', 'full_name', 'summary']], on='id', how='left')
candidate_skills_grouped = candidates_skills.merge(skills[['id', 'name']], left_on='skills_id', right_on='id', how='left')
candidate_skills_agg = candidate_skills_grouped.groupby('candidate_id')['name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
candidate_skills_agg.columns = ['candidate_id', 'skills_list']
df_candidates = candidates_full.merge(candidate_skills_agg, left_on='id', right_on='candidate_id', how='left')
df_candidates = df_candidates[['id', 'full_name', 'summary', 'education', 'expect_salary', 'skills_list']].copy()
df_candidates.columns = ['candidate_id', 'full_name', 'summary', 'education', 'expect_salary', 'skills']

# --- MERGE DATA FOR JOBS ---
industries_renamed = industries[['id', 'name']].rename(columns={'id': 'ind_id', 'name': 'industry'})
levels_renamed = levels[['id', 'name']].rename(columns={'id': 'lvl_id', 'name': 'level'})
job_types_renamed = job_types[['id', 'name']].rename(columns={'id': 'jt_id', 'name': 'job_type'})
job_posts_full = job_posts.merge(industries_renamed, left_on='industry_id', right_on='ind_id', how='left')
job_posts_full = job_posts_full.merge(levels_renamed, left_on='level_id', right_on='lvl_id', how='left')
job_posts_full = job_posts_full.merge(job_types_renamed, left_on='job_type_id', right_on='jt_id', how='left')
job_skills_grouped = job_posts_skills.merge(skills[['id', 'name']], left_on='skills_id', right_on='id', how='left')
job_skills_agg = job_skills_grouped.groupby('job_post_id')['name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
job_skills_agg.columns = ['job_post_id', 'skills_list']
df_jobs = job_posts_full.merge(job_skills_agg, left_on='id', right_on='job_post_id', how='left')
df_jobs = df_jobs[['id', 'title', 'description', 'salary', 'industry', 'level', 'job_type', 'skills_list']].copy()
df_jobs.columns = ['job_id', 'title', 'description', 'salary', 'industry', 'level', 'job_type', 'skills']

# Fillna & Convert Types
df_candidates.fillna('', inplace=True)
df_candidates['expect_salary'] = pd.to_numeric(df_candidates['expect_salary'], errors='coerce').fillna(0)
df_jobs.fillna('', inplace=True)
df_jobs['salary'] = pd.to_numeric(df_jobs['salary'], errors='coerce').fillna(0)

print(f"Candidates loaded: {len(df_candidates)}")
print(f"Jobs loaded: {len(df_jobs)}")

Candidates loaded: 55
Jobs loaded: 128


In [60]:
# 2.2 NLP Preprocessing Functions
STOPWORDS_PATH = "../data/nlp/vietnamese-stopwords.txt"
def load_vietnamese_stopwords(path: str) -> set:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return set(line.strip() for line in f if line.strip())
    except:
        return set()

vietnamese_stopwords = load_vietnamese_stopwords(STOPWORDS_PATH)

def preprocess_vietnamese_text(text: Any) -> str:
    if not text or pd.isna(text): return ""
    # Loại bỏ các thẻ HTML
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = text.lower()
    text = re.sub(r'\S+@\S+', ' ', text) # Remove email
    text = re.sub(r'http\S+', ' ', text) # Remove URL
    text = re.sub(r'[^a-zàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ0-9\s]', ' ', text)
    tokens = word_tokenize(text, format="text").split()
    clean_tokens = [t for t in tokens if t not in vietnamese_stopwords and not t.isdigit() and len(t) > 1]
    return " ".join(clean_tokens)

# Apply Preprocessing
print("Preprocessing Candidates...")
df_candidates['semantic_text'] = (df_candidates['summary'] + " " + df_candidates['education'] + " " + df_candidates['skills']).apply(preprocess_vietnamese_text)

print("Preprocessing Jobs...")
df_jobs['semantic_text'] = (df_jobs['title'] + " " + df_jobs['description'] + " " + df_jobs['skills']).apply(preprocess_vietnamese_text)

Preprocessing Candidates...
Preprocessing Jobs...


## 3. Vectorization & ChromaDB Setup
Đây là các thành phần cốt lõi sẽ được load một lần khi Server AI khởi động.

In [61]:
# 3.1 Load Embedding Model
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
embedding_model = SentenceTransformer(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model.to(device)
print(f"Model loaded: {model_name} on {device}")

Model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 on cpu


In [62]:
# 3.2 Setup ChromaDB Client
# [SỬA LẠI] Đường dẫn DB trỏ ra thư mục gốc của project (hwjob-ai)
DB_PATH = "../../chroma_db_data"

# Reset DB nếu cần (Cẩn thận!)
RESET_DB = True
if RESET_DB and os.path.exists(DB_PATH):
    print(f"Deleting old DB at {DB_PATH}...")
    try:
        shutil.rmtree(DB_PATH)
    except Exception as e:
        print(f"Error deleting DB: {e}")

chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Tạo 2 Collections
job_collection = chroma_client.get_or_create_collection(
    name="job_posts",
    metadata={"hnsw:space": "cosine"}
)

candidate_collection = chroma_client.get_or_create_collection(
    name="candidates",
    metadata={"hnsw:space": "cosine"}
)

print("Collections ready.")

Deleting old DB at ../chroma_db_data...
Error deleting DB: [WinError 32] The process cannot access the file because it is being used by another process: '../chroma_db_data\\9106fc3e-c80e-4150-b0f7-2088a02c866e\\data_level0.bin'
Collections ready.


## 4. Indexing Data (Lưu vào ChromaDB)

**Ghi chú trong kiến trúc thực tế:**
- Các cell dưới đây mô phỏng logic cho các API endpoint trên Server AI, ví dụ:
  - `POST /api/ai/index-job`
  - `POST /api/ai/index-candidate`
- Các API này được gọi bởi Server Spring Boot mỗi khi có Job/Candidate mới được tạo hoặc cập nhật.

In [63]:
# 4.1 Indexing Jobs (Cho luồng Candidate tìm Job)
if job_collection.count() == 0:
    print("Indexing Job Posts...")
    job_embeddings = embedding_model.encode(df_jobs['semantic_text'].tolist(), show_progress_bar=True)

    ids = df_jobs['job_id'].astype(str).tolist()
    documents = df_jobs['title'].tolist()
    metadatas = []
    for _, row in df_jobs.iterrows():
        metadatas.append({
            "skills": str(row['skills']),
            "level": str(row['level'])
        })

    job_collection.add(ids=ids, embeddings=job_embeddings.tolist(), metadatas=metadatas, documents=documents)
    print(f"Indexed {len(ids)} jobs.")
else:
    print("Jobs already indexed.")

# 4.2 Indexing Candidates (Cho luồng Nhà tuyển dụng tìm Candidate)
if candidate_collection.count() == 0:
    print("Indexing Candidates...")
    cand_embeddings = embedding_model.encode(df_candidates['semantic_text'].tolist(), show_progress_bar=True)

    ids = df_candidates['candidate_id'].astype(str).tolist()
    documents = df_candidates['full_name'].tolist()
    metadatas = []
    for _, row in df_candidates.iterrows():
        metadatas.append({
            "skills": str(row['skills']),
        })

    candidate_collection.add(ids=ids, embeddings=cand_embeddings.tolist(), metadatas=metadatas, documents=documents)
    print(f"Indexed {len(ids)} candidates.")
else:
    print("Candidates already indexed.")

Jobs already indexed.
Candidates already indexed.


## 5. Recommendation Logic (Optimized)

**Ghi chú trong kiến trúc thực tế:**
- Các hàm `recommend_jobs` và `rank_pending_candidates` là logic cốt lõi cho các API:
  - `POST /api/ai/recommend-jobs`
  - `POST /api/ai/rank-pending-candidates`
- Input của các hàm này trong thực tế sẽ là một JSON payload do Spring Boot gửi đến.
- Output (danh sách ID và score) sẽ được trả về cho Spring Boot để xử lý tiếp.

In [64]:
# ==================== OPTIMIZATION #1 (REFACTORED): Scalable Dynamic Weighting ====================
# LOCATION: Cell 5
# IMPACT: Cấu trúc lại để dễ dàng mở rộng với các level mới mà không cần sửa logic hàm.
# Bảng ánh xạ trọng số. Dễ dàng thêm level mới (vd: 'principal') ở đây.
WEIGHT_MAP = {
    # Từ khóa: {trọng số skill, trọng số semantic}
    'intern': {'skill': 0.3, 'semantic': 0.7},
    'thực tập': {'skill': 0.3, 'semantic': 0.7},
    'fresher': {'skill': 0.4, 'semantic': 0.6},
    'junior': {'skill': 0.5, 'semantic': 0.5},
    'senior': {'skill': 0.7, 'semantic': 0.3},
    'manager': {'skill': 0.6, 'semantic': 0.4},
    'trưởng phòng': {'skill': 0.6, 'semantic': 0.4},
    'lead': {'skill': 0.7, 'semantic': 0.3}, # Thêm ví dụ
}
DEFAULT_WEIGHTS = {'skill': 0.5, 'semantic': 0.5}

def get_dynamic_weights(job_level: str) -> Dict[str, float]:
    """
    Trả về trọng số phù hợp theo cấp độ công việc bằng cách tra cứu trong WEIGHT_MAP.
    Hàm này có khả năng mở rộng cao.
    """
    level_lower = str(job_level).lower()

    # Duyệt qua bảng ánh xạ để tìm từ khóa phù hợp
    for keyword, weights in WEIGHT_MAP.items():
        if keyword in level_lower:
            return weights

    # Nếu không tìm thấy từ khóa nào, trả về trọng số mặc định
    return DEFAULT_WEIGHTS

### 5.1 Flow 1: Candidate tìm Job (recommend_jobs)
def calculate_job_score(semantic_score: float, job_meta: Dict, candidate_profile: Dict) -> tuple:
    # Robust Skill Matching & Zero Division Protection
    job_skills = set([s.strip().lower() for s in str(job_meta.get('skills', '')).split(',') if s.strip()])
    cand_skills = set([s.strip().lower() for s in str(candidate_profile.get('skills', '')).split(',') if s.strip()])

    if not job_skills:
        skill_score = 0.0
    else:
        match_count = len(job_skills.intersection(cand_skills))
        skill_score = match_count / len(job_skills)

    # Score Normalization
    skill_score = max(0.0, min(1.0, skill_score))
    semantic_score = max(0.0, min(1.0, semantic_score))

    # Lấy trọng số động dựa trên level của job
    job_level = job_meta.get('level', 'Junior')
    weights = get_dynamic_weights(job_level)

    final_score = (weights['skill'] * skill_score) + (weights['semantic'] * semantic_score)
    return final_score, skill_score

def recommend_jobs(candidate_profile_data: Dict, top_k: int = 10) -> pd.DataFrame:
    try:
        # Tạo semantic text từ payload nhận được
        semantic_text = (str(candidate_profile_data.get('summary', '')) + " " +
                         str(candidate_profile_data.get('education', '')) + " " +
                         str(candidate_profile_data.get('skills', '')))

        query_text = preprocess_vietnamese_text(semantic_text)
        query_vector = embedding_model.encode(query_text).tolist()

        n_results = min(50, top_k * 3)

        results = job_collection.query(query_embeddings=[query_vector], n_results=n_results, include=["metadatas", "documents", "distances"])

        ranked = []
        ids, metas, dists, titles = results['ids'][0], results['metadatas'][0], results['distances'][0], results['documents'][0]

        for i in range(len(ids)):
            semantic_score = 1 - dists[i]
            final_score, skill_score = calculate_job_score(semantic_score, metas[i], candidate_profile_data)
            ranked.append({
                "job_id": ids[i], "title": titles[i],
                "final_score": round(final_score, 3),
                "semantic": round(semantic_score, 3),
                "skill_match": round(skill_score, 3),
                "level": metas[i].get('level'),
                "skills": metas[i]['skills']
            })

        ranked.sort(key=lambda x: x['final_score'], reverse=True)
        return pd.DataFrame(ranked[:top_k])
    except Exception as e:
        print(f"Lỗi trong recommend_jobs: {e}")
        return pd.DataFrame()


In [65]:
### 5.2 Flow 2: Sắp xếp các Ứng viên đã Apply (rank_pending_candidates)
def calculate_candidate_score(semantic_score: float, cand_meta: Dict, job_data: Dict) -> tuple:
    # Robust Skill Matching & Zero Division Protection
    cand_skills = set([s.strip().lower() for s in str(cand_meta.get('skills', '')).split(',') if s.strip()])
    job_skills = set([s.strip().lower() for s in str(job_data.get('skills', '')).split(',') if s.strip()])

    if not job_skills:
        skill_score = 0.0
    else:
        match_count = len(job_skills.intersection(cand_skills))
        skill_score = match_count / len(job_skills)

    # Score Normalization
    skill_score = max(0.0, min(1.0, skill_score))
    semantic_score = max(0.0, min(1.0, semantic_score))

    # Lấy trọng số động dựa trên level của job
    job_level = job_data.get('level', 'Junior')
    weights = get_dynamic_weights(job_level)

    final_score = (weights['skill'] * skill_score) + (weights['semantic'] * semantic_score)
    return final_score, skill_score

def rank_pending_candidates(job_post_data: Dict) -> pd.DataFrame:
    try:
        # 1. Lấy danh sách ID ứng viên "pending" từ payload
        pending_ids = job_post_data.get('pending_candidate_ids', [])
        if not pending_ids:
            return pd.DataFrame()

        # 2. Tạo Query Vector cho Job Post
        job_semantic_text = (str(job_post_data.get('title', '')) + " " +
                             str(job_post_data.get('description', '')) + " " +
                             str(job_post_data.get('skills', '')))
        job_query_text = preprocess_vietnamese_text(job_semantic_text)
        job_vector = embedding_model.encode(job_query_text)

        # 3. Lấy thông tin của CHỈ những ứng viên pending từ ChromaDB
        pending_ids_str = [str(id) for id in pending_ids]
        candidates_data = candidate_collection.get(ids=pending_ids_str, include=["metadatas", "documents", "embeddings"])

        # 4. Chấm điểm và sắp xếp lại danh sách này
        ranked = []
        cand_ids, cand_metas, cand_embeds, cand_names = candidates_data['ids'], candidates_data['metadatas'], candidates_data['embeddings'], candidates_data['documents']

        for i in range(len(cand_ids)):
            cand_vector = np.array(cand_embeds[i])

            # Tính cosine similarity giữa vector của job và vector của candidate
            similarity = np.dot(job_vector, cand_vector) / (np.linalg.norm(job_vector) * np.linalg.norm(cand_vector))
            semantic_score = float(similarity)

            final_score, skill_score = calculate_candidate_score(semantic_score, cand_metas[i], job_post_data)

            ranked.append({
                "candidate_id": cand_ids[i], "full_name": cand_names[i],
                "final_score": round(final_score, 3),
                "semantic": round(semantic_score, 3),
                "skill_match": round(skill_score, 3),
                "skills": cand_metas[i]['skills']
            })

        # 5. Sắp xếp các ứng viên đã apply theo điểm số
        ranked.sort(key=lambda x: x['final_score'], reverse=True)
        return pd.DataFrame(ranked)
    except Exception as e:
        print(f"Lỗi trong rank_pending_candidates: {e}")
        return pd.DataFrame()

## 6. Demo & Testing (Updated)
Đây là bước kiểm thử logic AI, mô phỏng việc Server AI nhận request từ Spring Boot.

In [66]:
# TEST 1: Candidate tìm Job
print("--- TEST 1: Candidate tìm Job ---")
# 1. Spring Boot lấy thông tin ứng viên từ SQL DB
if not df_candidates.empty:
    sample_cand_id = df_candidates.iloc[0]['candidate_id']
    sample_candidate_profile = df_candidates[df_candidates['candidate_id'] == sample_cand_id].to_dict('records')[0]

    print(f"Input Payload (từ Spring Boot):")
    print(sample_candidate_profile)

    # 2. Spring Boot gửi payload này đến Server AI, Server AI xử lý
    recs_jobs = recommend_jobs(sample_candidate_profile)

    print("\nOutput (Danh sách Job ID trả về cho Spring Boot):")
    display(recs_jobs)
else:
    print("Không có dữ liệu ứng viên để test.")

# TEST 2: Sắp xếp các Ứng viên đã Apply
print("\n--- TEST 2: Sắp xếp các Ứng viên đã Apply ---")
# Tạo kịch bản test từ dữ liệu thực tế trong applications.csv
try:
    applications = pd.read_csv("../data/csv_data/applications.csv")

    if not applications.empty:
        # 1. Tìm job_post_id có nhiều ứng viên apply nhất
        most_applied_job_id = applications['job_post_id'].value_counts().idxmax()

        # 2. Lấy danh sách candidate_id đã apply vào job đó
        pending_list = applications[applications['job_post_id'] == most_applied_job_id]['candidate_id'].tolist()

        # 3. Lấy thông tin của job post đó
        job_info_list = df_jobs[df_jobs['job_id'] == most_applied_job_id].to_dict('records')
        if job_info_list:
            sample_job_post = job_info_list[0]
            sample_job_post['pending_candidate_ids'] = pending_list

            print(f"Input Payload (từ Spring Boot):")
            # In một phần để tránh quá dài
            print({k: v for k, v in sample_job_post.items() if k != 'description'})
            print(f"\n[INFO] Sẽ chấm điểm và sắp xếp các ứng viên có ID: {pending_list}")

            # 4. Spring Boot gửi payload này đến Server AI, Server AI xử lý
            ranked_cands = rank_pending_candidates(sample_job_post)

            print("\nOutput (Danh sách Candidate đã được sắp xếp theo mức độ phù hợp):")
            display(ranked_cands)
        else:
            print(f"Không tìm thấy thông tin cho Job ID: {most_applied_job_id}")
    else:
        print("File applications.csv rỗng, không có dữ liệu để test.")

except FileNotFoundError:
    print("Không tìm thấy file applications.csv. Bỏ qua Test 2.")
except Exception as e:
    print(f"Đã xảy ra lỗi khi xử lý Test 2: {e}")

--- TEST 1: Candidate tìm Job ---
Input Payload (từ Spring Boot):
{'candidate_id': '015adece-ab49-460d-904c-0c9e59b4d67d', 'full_name': 'Đỗ Ngọc Anh', 'summary': 'Trợ lý hành chính. Thành thạo tin học văn phòng, soạn thảo văn bản, lịch trình. Cẩn thận, ngăn nắp.', 'education': 'Cao đẳng Hành chính - Văn thư Lưu trữ', 'expect_salary': 9500000.0, 'skills': 'Virtual Assistance, Microsoft Office, Time Management', 'semantic_text': 'trợ_lý hành_chính thành_thạo tin_học văn_phòng soạn_thảo văn_bản lịch_trình cẩn_thận ngăn_nắp cao_ẳng hành_chính văn_thư lưu_trữ virtual assistance microsoft office time management'}

Output (Danh sách Job ID trả về cho Spring Boot):


,job_id,title,final_score,semantic,skill_match,level,skills
0,5f4c4a65-fb86-49d5-8579-1bcefa416cb5,Trợ lý Hành chính Online (Part-time),0.977,0.953,1.000,None,"Virtual Assistance, Microsoft Office, Time Management"
1,c9dd3da0-a3df-4d70-878d-72fbe9159d7d,Trợ lý Giám đốc Dự án (Project Assistant),0.854,0.957,0.750,None,"Project Management, Virtual Assistance, Microsoft Office, Time Management"
2,38a3b0e4-7274-42f4-bf38-e1c065ffa368,Trợ lý Ảo (Virtual Assistant) cho Doanh nhân,0.719,0.772,0.667,None,"Tiếng Anh, Virtual Assistance, Time Management"
3,af5be735-5099-413a-8925-d0cc08d4637c,Lái xe Tải Container Đường dài,0.717,0.434,1.000,None,Time Management
4,616d518e-4988-419e-b243-562bbba14e83,Cộng tác viên Nhập liệu Số hóa Sách cũ,0.567,0.634,0.500,None,"Data Entry, Microsoft Office"
5,9fecfc40-fb2d-4dfa-b844-6e8711cd08d4,Đánh máy Bản thảo Tiểu thuyết,0.553,0.606,0.500,None,"Data Entry, Microsoft Office"
6,069a148a-502a-4104-a9ef-39361b1a5a49,Số hóa Biểu mẫu & Phiếu khảo sát cũ,0.551,0.602,0.500,None,"Data Entry, Microsoft Office"
7,9a0fbff1-98d7-49e7-b692-353b17344a99,Nhập liệu Online tại nhà (Freelance),0.551,0.601,0.500,None,"Data Entry, Microsoft Office"
8,ed5983e5-6adc-48a7-a139-3f0fdd83cc65,Điều dưỡng viên Khoa Nội tổng quát,0.550,0.601,0.500,None,"Customer Service, Time Management"
9,5e85fed6-2961-4825-9004-d17f4a50ab7d,Kỹ sư Vận hành Nhà máy Điện,0.505,0.509,0.500,None,"Critical Thinking, Time Management"



--- TEST 2: Sắp xếp các Ứng viên đã Apply ---
Input Payload (từ Spring Boot):
{'job_id': '8009199f-531c-4b05-9b6d-23adc531fd62', 'title': 'Kỹ sư Dữ liệu (Data Engineer)', 'salary': 38000000.0, 'industry': 'Bán lẻ & Thương mại điện tử', 'level': 'Senior', 'job_type': 'Toàn thời gian', 'skills': 'Python, SQL, Docker', 'semantic_text': 'kỹ_sư dữ_liệu data_engineer xây_dựng vận_hành data pipeline nền_tảng cloud aws gcp tiki công_nghệ airflow spark kafka bigquery redshift python scala mô_tả thiết_kế data warehouse etl elt pipeline phục_vụ phân_tích machine learning python sql docker', 'pending_candidate_ids': ['4fed3b64-e939-475f-8677-f8a4dc2d7e7e', 'bc5d7c15-41b0-4b27-a71f-57a8f301fb3f']}

[INFO] Sẽ chấm điểm và sắp xếp các ứng viên có ID: ['4fed3b64-e939-475f-8677-f8a4dc2d7e7e', 'bc5d7c15-41b0-4b27-a71f-57a8f301fb3f']

Output (Danh sách Candidate đã được sắp xếp theo mức độ phù hợp):


,candidate_id,full_name,final_score,semantic,skill_match,skills
0,bc5d7c15-41b0-4b27-a71f-57a8f301fb3f,Đỗ Ngọc Lan,0.641,0.580,0.667,"Python, SQL"
1,4fed3b64-e939-475f-8677-f8a4dc2d7e7e,Hồ Sỹ Đức,0.595,0.429,0.667,"SQL, Docker, Project Management"
